# Advanced RAG — Ingestion Pipeline (PDF / DOCX / TXT)

**Goal:** A local, reusable ingestion notebook: load → parse → normalize → identify sources → detect changes → chunk → identify chunks → inspect duplicates → embed → synchronize Qdrant → update manifest.

Run cells **top to bottom**. Each stage is separated so you can inspect outputs and ask questions. We deliberately do **not** implement retrieval here.

**Dataset:** Put your sample `.pdf`, `.docx`, and `.txt` files inside `data/`. The ingestion logic depends on file extensions, not on particular filenames. This first version handles *textual* PDFs, DOCX and TXT; OCR and tables are an optional extension (described below).

## 1 — Paths and configuration

Qdrant is stored locally in `storage/qdrant/`; SQLite tracks ingested source files in `storage/manifest.sqlite3`. Set `DATA_DIR` to your folder.

In [10]:
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
DATA_DIR = Path("data").resolve()
STORE_DIR = Path("storage")
STORE_DIR.mkdir(exist_ok=True)
QDRANT_PATH = str((STORE_DIR / "qdrant").resolve())
DB_PATH = STORE_DIR / "manifest.sqlite3"
COLLECTION = "documents"
EMBEDDING_MODEL = "gemini-embedding-001"
EMBEDDING_DIM = "1024"
CHUNK_SIZE = 500       # tokens
CHUNK_OVERLAP = 75     # tokens
SUPPORTED = {".pdf", ".docx", ".txt"}

files = sorted(p for p in DATA_DIR.rglob("*") if p.is_file() and p.suffix.lower() in SUPPORTED) if DATA_DIR.exists() else []
print(f"Found {len(files)} supported files:", *[str(p.relative_to(DATA_DIR)) for p in files], sep="\n  ")

Found 3 supported files:
  company_handbook.pdf
  employee_faq.txt
  leave_policy.docx


## 2 — Source-specific loading

LangChain loaders produce `Document(page_content=..., metadata=...)` objects. PDF may produce **one Document per page**. We retain page numbers for citations later. A malformed PDF may warn about `startxref`; inspect its extracted text before proceeding.

In [11]:
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader

def load_file(path: Path):
    """returns LOADED DOCS of single file by path """
    ext = path.suffix.lower()
    if ext == ".pdf":
        return PyPDFLoader(str(path)).load()
    if ext == ".docx":
        return Docx2txtLoader(str(path)).load()
    if ext == ".txt":
        return TextLoader(str(path), encoding="utf-8", autodetect_encoding=True).load()
    raise ValueError(f"Unsupported file: {path}")

# Sanity-check ONE source first
if files:
    sample_docs = load_file(files[0])
    print("Documents/pages:", len(sample_docs))
    print("Metadata:", sample_docs[0].metadata)
    print("Text preview:", repr(sample_docs[0].page_content[:400]))

incorrect startxref pointer(1)
parsing for Object Streams


Documents/pages: 2
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-30T06:38:55+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-07-30T06:38:55+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'C:\\Users\\ESHAAN\\HAKUR\\AI-GOAT\\RagMail\\data\\company_handbook.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}
Text preview: 'COMPANY EMPLOYEE HANDBOOK\n Company Name: TechNova Solutions\n Founded: 2020\n Industry: Artificial Intelligence and Software Development\n1. WORKING HOURS\nThe standard working hours are from 9:00 AM to 6:00 PM, Monday to Friday.\nEmployees are expected to arrive on time and complete their assigned working hours.\nEmployees may work remotely when approved by their reporting manager.\n2. WORK FROM HOME PO'


## 3 — Complex extraction / OCR: optional extension point

For complicated PDFs, **replace only the `.pdf` branch** of `load_file()` with Docling or PyMuPDF4LLM. Scanned pages need OCR. Keep the output contract as a list of LangChain `Document` objects with page/source metadata.

**Not automatically enabled:** OCR changes dependencies and can be slow; first test the simple PDFs. DOCX tables may also require a richer parser.

## 4 — Normalize extracted text

`ftfy` repairs common broken Unicode. Simple whitespace cleanup preserves paragraphs and newlines. Some technical documents need strict symbol preservation; test the output against the original.

In [12]:
import re
import ftfy
from langchain_core.documents import Document

def normalize_text(s: str) -> str:
    s = ftfy.fix_text(s).replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[^\S\n]+", " ", s)
    s = re.sub(r" *\n *", "\n", s)
    return re.sub(r"\n{3,}", "\n\n", s).strip()

if files:
    print(repr(normalize_text(sample_docs[0].page_content[:400])))

'COMPANY EMPLOYEE HANDBOOK\nCompany Name: TechNova Solutions\nFounded: 2020\nIndustry: Artificial Intelligence and Software Development\n1. WORKING HOURS\nThe standard working hours are from 9:00 AM to 6:00 PM, Monday to Friday.\nEmployees are expected to arrive on time and complete their assigned working hours.\nEmployees may work remotely when approved by their reporting manager.\n2. WORK FROM HOME PO'


## 5 — Standardize IDs and metadata

The **source ID** is stable across edits: SHA-256 of its path relative to the dataset root. The **source hash** is SHA-256 of raw file bytes, so it changes when the file changes. Each extracted page receives a distinct part index. Do not put absolute machine-specific paths in IDs.

In [13]:
import hashlib

def digest(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()

def source_key(path: Path) -> str:
    return path.relative_to(DATA_DIR).as_posix()

def source_id(path: Path) -> str:
    return digest(source_key(path).encode("utf-8"))

def file_hash(path: Path) -> str:
    return digest(path.read_bytes())

def standardized_docs(path: Path) -> list[Document]:
    """takes Path, loads DOCS with respective DATALOADER, makes a standar Metadata for the DOC and Normalises page Content"""
    sid = source_id(path)
    pieces = load_file(path)
    result = []
    for i, original_doc in enumerate(pieces):
        meta = {"source_id": sid, "source_path": source_key(path),
                "file_type": path.suffix.lower(), "part_index": i,
                "page": original_doc.metadata.get("page")}
        result.append(Document(page_content=normalize_text(original_doc.page_content), metadata=meta))
    return result

if files:
    print("Source ID:", source_id(files[0]))
    print("Content SHA256:", file_hash(files[0]))
    print(standardized_docs(files[0])[0].metadata)

incorrect startxref pointer(1)
parsing for Object Streams


Source ID: 381006df658fc28f9148ee970934dbbe87c092d982f9eaa9fd092b3801fb2793
Content SHA256: 207c2f4cfd81d4586152f40b628a7f111166219591970425daf94c9c01e49313
{'source_id': '381006df658fc28f9148ee970934dbbe87c092d982f9eaa9fd092b3801fb2793', 'source_path': 'company_handbook.pdf', 'file_type': '.pdf', 'part_index': 0, 'page': 0}


## 6 — SQLite manifest for incremental ingestion

A manifest answers: *Has this source been indexed? Has it changed? Which Qdrant point IDs belong to it?* We store source hashes and a JSON list of chunk IDs. The manifest is updated **only after Qdrant succeeds**.

In [ ]:
import sqlite3, json

def connect_manifest():
    conn = sqlite3.connect(DB_PATH)
    conn.execute("""CREATE TABLE IF NOT EXISTS sources (
        source_id TEXT PRIMARY KEY,
        source_path TEXT NOT NULL,
        sha256 TEXT NOT NULL,
        chunk_ids_json TEXT NOT NULL
    )""")
    conn.commit()
    return conn

conn = connect_manifest()

def existing_manifest(conn):
    rows = conn.execute("SELECT source_id, source_path, sha256, chunk_ids_json FROM sources").fetchall()
    return {r[0]: {"path": r[1], "hash": r[2], "ids": json.loads(r[3])} for r in rows}

manifest = existing_manifest(conn)
changed_files = [p for p in files if source_id(p) not in manifest or manifest[source_id(p)]["hash"] != file_hash(p)]
removed_ids = set(manifest) - {source_id(p) for p in files}
print(f"New/modified: {len(changed_files)} | Unchanged: {len(files)-len(changed_files)} | Removed: {len(removed_ids)}")

New/modified: 3 | Unchanged: 0 | Removed: 0


## 7 — Token-aware chunking

`RecursiveCharacterTextSplitter.from_tiktoken_encoder` counts tokens instead of plain characters, preserving paragraph boundaries when possible. It's a practical approximate tokenizer for chunk sizing; the Gemini model can tokenize differently. We chunk **within** a page/part, not across source files.

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

def chunk_source(path: Path) -> list[Document]:
    """takes Path, loads DOCS with respective DATALOADER, makes a standar Metadata for the DOC and Normalises page Content, THEN CHUNKS and returns CHUNK DOC ARR"""
    result = []
    for std_doc in standardized_docs(path):
        for index, chunk in enumerate(splitter.split_documents([std_doc])):
            # Deterministic ID: same source, part, index, and text -> same ID
            key = f"{std_doc.metadata['source_id']}:{std_doc.metadata['part_index']}:{index}:{chunk.page_content}"
            chunk.id = digest(key.encode("utf-8"))
            chunk.metadata["chunk_index"] = index
            result.append(chunk)
    return result

if changed_files:
    preview_chunks = chunk_source(changed_files[0])
    print("Chunk count:", len(preview_chunks))
    print("ID:", preview_chunks[0].id if preview_chunks else "(no text)")
    print("Preview:", preview_chunks[0].page_content[:300] if preview_chunks else "(empty)")

incorrect startxref pointer(1)
parsing for Object Streams


Chunk count: 2
ID: 668f4de56868f1e0853d0a1b4b3731b4adf329d6f935190d491e3f1a0eb02e7b
Preview: COMPANY EMPLOYEE HANDBOOK
Company Name: TechNova Solutions
Founded: 2020
Industry: Artificial Intelligence and Software Development
1. WORKING HOURS
The standard working hours are from 9:00 AM to 6:00 PM, Monday to Friday.
Employees are expected to arrive on time and complete their assigned working 


## 8 — Deduplication checks

**Exact:** SHA-256 of normalized chunk text. **Near duplicate:** similarity ratio via RapidFuzz. This version *reports* cross-source duplicates rather than silently dropping them, because dropping duplicate chunks could remove a source's citation/provenance. Very large datasets need an indexed near-duplicate approach (e.g., MinHash/LSH).

In [16]:
from collections import defaultdict
from rapidfuzz.fuzz import ratio

staged = {source_id(p): chunk_source(p) for p in changed_files}
all_chunks = [c for chunks in staged.values() for c in chunks if c.page_content]
by_hash = defaultdict(list)
for c in all_chunks:
    by_hash[digest(c.page_content.encode("utf-8"))].append(c.id)
exact_groups = [ids for ids in by_hash.values() if len(ids) > 1]
print("Exact duplicate text groups (within changed files):", len(exact_groups))

# Small inspection only; do not do all-pairs comparisons on big datasets.
example_near = []
for i, a in enumerate(all_chunks[:30]):
    for b in all_chunks[i+1:30]:
        if a.id != b.id and a.page_content != b.page_content:
            score = ratio(a.page_content, b.page_content)
            if score >= 90:
                example_near.append((round(score, 1), a.id[:10], b.id[:10]))
print("Near-duplicate examples (>90%):", example_near[:5])

incorrect startxref pointer(1)
parsing for Object Streams


Exact duplicate text groups (within changed files): 0
Near-duplicate examples (>90%): []


## 9 — Embeddings: Gemini

This is the **first cell that uses a paid/external API**. Check `GOOGLE_API_KEY` is available. The embedding dimensionality is inferred from the first embedding, preventing hard-coded mismatches. For larger datasets, batch/rate-limit requests and estimate cost before indexing.

In [17]:
import os
from langchain_google_genai import GoogleGenerativeAIEmbeddings

if not os.getenv("GOOGLE_API_KEY"):
    print("Set GOOGLE_API_KEY in your environment/.env before continuing with embedding + Qdrant cells.")
else:
    embeddings = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL)
    print("Embedding setup ready. Vector dimensions:", len(embeddings.embed_query("dimension probe")))

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Embedding setup ready. Vector dimensions: 3072


## 10 — Persistent Qdrant collection

Qdrant uses a **local path** so no Docker/server is required. A single writer should own this path; close other notebooks or processes using it. Payload stores source metadata and chunk text for retrieval and citations later.

In [9]:
from qdrant_client import QdrantClient, models

# Run after embeddings have been initialized in Step 9.
assert "embeddings" in globals(), "Configure GOOGLE_API_KEY and re-run Step 9."
client = QdrantClient(path=QDRANT_PATH)
vector_size = len(embeddings.embed_query("dimension probe"))
if not client.collection_exists(COLLECTION):
    client.create_collection(
        collection_name=COLLECTION,
        vectors_config=models.VectorParams(size=vector_size, distance=models.Distance.COSINE)
    )
else:
    existing_size = client.get_collection(COLLECTION).config.params.vectors.size
    assert existing_size == vector_size, "Existing collection embedding dimensions differ. Use a new collection."
print("Qdrant collection ready:", COLLECTION)

Qdrant collection ready: documents


## 11 — Synchronize new, modified and deleted sources

Upsert current chunks and remove outdated point IDs. For modified files, **upsert first**, then delete old points no longer present. For removed files, delete all their points and manifest entries. Chunk IDs are 64-digit SHA hex strings, but Qdrant point IDs require UUIDs or integers; convert deterministically with UUIDv5.

**Safety:** files missing from `DATA_DIR` are treated as deleted. Make sure `DATA_DIR` points at the entire intended dataset before executing.

In [18]:
from uuid import uuid5, NAMESPACE_URL

def point_id(chunk_id: str) -> str:
    """Give Each Chunk a reproducable unique ID"""
    return str(uuid5(NAMESPACE_URL, chunk_id))

def delete_chunk_ids(ids: list[str]):
    """Delete Chunks from VectorDB on basis of Chunk Ids"""
    if ids:
        client.delete(collection_name=COLLECTION, points_selector=models.PointIdsList(points=[point_id(i) for i in ids]), wait=True)

def ingest_one(path: Path):
    sid = source_id(path)
    chunks = staged[sid]
    previous_ids = set(manifest.get(sid, {}).get("ids", []))
    new_ids = [c.id for c in chunks]
    texts = [c.page_content for c in chunks]
    # Embed only when there is nonempty content
    vectors = embeddings.embed_documents(texts) if texts else []
    for start in range(0, len(chunks), 64):
        points = [models.PointStruct(
            id=point_id(c.id), vector=v,
            payload={"text": c.page_content, **c.metadata, "chunk_id": c.id}
        ) for c, v in zip(chunks[start:start+64], vectors[start:start+64])]
        if points:
            client.upsert(collection_name=COLLECTION, points=points, wait=True)
    delete_chunk_ids(list(previous_ids - set(new_ids)))
    with conn:
        conn.execute("INSERT OR REPLACE INTO sources VALUES (?, ?, ?, ?)",
                     (sid, source_key(path), file_hash(path), json.dumps(new_ids)))
    print("Indexed:", source_key(path), "chunks:", len(new_ids))

for path in changed_files:
    ingest_one(path)

for sid in removed_ids:
    delete_chunk_ids(manifest[sid]["ids"])
    with conn:
        conn.execute("DELETE FROM sources WHERE source_id=?", (sid,))
    print("Removed source:", manifest[sid]["path"])

Indexed: company_handbook.pdf chunks: 2
Indexed: employee_faq.txt chunks: 1
Indexed: leave_policy.docx chunks: 1


## 12 — Validate ingestion and incremental updates

Run this after indexing, and again after editing/deleting/adding one test file. Re-running without any changes should show **0 new/modified** in Step 6 (you must rerun Steps 1, 6, 8, and 11 to refresh notebook variables; preferably restart and run all).

In [19]:
latest = existing_manifest(conn)
print("Manifest sources:", len(latest))
print("Manifest chunks:", sum(len(v["ids"]) for v in latest.values()))
print("Qdrant points:", client.count(collection_name=COLLECTION, exact=True).count)
assert client.count(collection_name=COLLECTION, exact=True).count == sum(len(v["ids"]) for v in latest.values()),     "Mismatch: investigate partial writes, duplicated IDs, or external modifications"
print("Validation passed")

Manifest sources: 3
Manifest chunks: 4
Qdrant points: 4
Validation passed


## 13 — Experiments you should try

1. Re-run unchanged: verify **zero** API embedding work and unchanged Qdrant point count.
2. Edit a TXT file: verify source hash changes and stale chunks disappear.
3. Delete one source file: verify its manifest record and Qdrant points disappear.
4. Add a duplicate TXT file: inspect exact duplicate report; verify both source citations remain possible.
5. Put a scanned PDF into `data/`: check whether the basic loader returns empty text, then add OCR **only** to the PDF loader.

**Next pipeline:** query embeddings → top-K search → filtering → cross-encoder reranking → context budgeting → grounded answers. Those belong in a separate retrieval notebook.

**Production caveat:** SQLite and Qdrant are distinct stores (no shared transaction); a process crash during synchronization may require reconciliation. This notebook is a learning baseline, not a production-atomic ingestion service.